# Topic embeddings with SGNS
Waller & Anderson (2021), *Quantifying social organization and political polarization in online platforms*, Methods: “Creating the community embedding”, p. 6 of the supplied PDF. [Paper](https://doi.org/10.1038/s41586-021-04167-x).

Topics are **words**, authors are **contexts**, and every comment contributes one pair (including repeated topic–author pairs). No text, sliding context window, or user–topic deduplication is used. This uses the authors' SGNS engine, [word2vecf](https://github.com/BIU-NLP/word2vecf), with 150 dimensions, 35 negatives, alpha 0.18, sample 0.0043 and globally shuffled pairs. Negative contexts follow the engine's frequency^0.75 sampler; its subsampling applies to both words and contexts.

We retain this project's top 1,000 leaf topics by unique users, rather than the paper's 10,006 subreddits by comments. These hyperparameters were tuned on Reddit, not validated on derStandard. One training pass is the engine default; the paper does not specify the number of passes. Four threads speed up training but are not bitwise deterministic. Set `THREADS = 1` for repeatable execution on the same build.

Comments are read in chunks. Compact integer pairs live in a memory-mapped file, are shuffled globally, then streamed into the native trainer. Intermediate pairs and user vocabularies are temporary. The engine also needs its fixed ~400 MB negative-sampling table, vocabulary hashes and user vectors.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from collections import Counter
import json
import subprocess
import numpy as np
import pandas as pd

DATA_DIR = Path("my_data")
RAW_DATA_DIR = Path("data")
USE_RAW_COMMENTS = True  # Exported comments cover only 945 selected topics.
OUTPUT_DIR = DATA_DIR / "sgns"
TOOL_DIR = Path(".tools/word2vecf")
TOOL_COMMIT = "2d503b095546660e0bb8b0594c7b442c7c533e49"
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
CHUNK_SIZE = 250_000
DIMENSIONS, NEGATIVE, ALPHA, SAMPLE = 150, 35, 0.18, 0.0043
EPOCHS, THREADS, SHUFFLE_SEED = 1, 4, 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Build the pinned native trainer
Requires Git and a C compiler (`cc`). Dependencies stay in `.tools/word2vecf`; Python only needs NumPy and pandas. Compiler and training output go to log files.

In [ ]:
if not TOOL_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/BIU-NLP/word2vecf.git", str(TOOL_DIR)], check=True)
    subprocess.run(["git", "-C", str(TOOL_DIR), "checkout", TOOL_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(TOOL_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != TOOL_COMMIT:
    raise ValueError("word2vecf version differs from the pinned implementation")
trainer = (TOOL_DIR / "word2vecf").resolve()
with (OUTPUT_DIR / "build.log").open("w") as log:
    subprocess.run(["cc", "-O3", "-pthread", *[str(TOOL_DIR / name) for name in
                    ("word2vecf.c", "vocab.c", "io.c")], "-lm", "-o", str(trainer)],
                   stdout=log, stderr=subprocess.STDOUT, check=True)

## 2. Resolve the top 1,000 topic paths
Keep IDs as strings, including leading zeros. Normalize empty topic levels and whitespace. Reject duplicate paths, conflicting article assignments, and unexpected topics rather than silently changing the training population.

In [ ]:
ranked_topics_df = pd.read_csv(DATA_DIR / "ranked_topics.csv", keep_default_na=False,
                              dtype={column: "string" for column in TOPIC_COLUMNS})
ranked_topics_df[TOPIC_COLUMNS] = ranked_topics_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())
topics_df = ranked_topics_df.sort_values(
    ["Unique_users", *TOPIC_COLUMNS], ascending=[False, True, True, True]
).head(1_000).reset_index(drop=True)
if len(topics_df) != 1_000 or topics_df[TOPIC_COLUMNS].duplicated().any():
    raise ValueError("Expected exactly 1,000 distinct topic paths")
if not topics_df["Is_Leaf"].eq(True).all() or not topics_df["Unique_users"].gt(0).all():
    raise ValueError("Expected active leaf topics")
topics_df.insert(0, "Topic_id", np.arange(len(topics_df)))
articles_df = pd.read_csv(DATA_DIR / "articles_top_1000.csv", dtype="string",
                          keep_default_na=False, usecols=["Article_Id", *TOPIC_COLUMNS])
articles_df[TOPIC_COLUMNS] = articles_df[TOPIC_COLUMNS].apply(lambda s: s.str.strip())
articles_df = articles_df.drop_duplicates()
if articles_df["Article_Id"].duplicated().any():
    raise ValueError("Conflicting topic paths for an Article_Id")
article_mapping = articles_df.merge(topics_df[["Topic_id", *TOPIC_COLUMNS]],
                                    on=TOPIC_COLUMNS, how="left", validate="many_to_one")
if article_mapping["Topic_id"].isna().any():
    raise ValueError("Selected articles contain a topic outside the top 1,000")
article_to_topic = article_mapping.set_index("Article_Id")["Topic_id"]
del articles_df, article_mapping

## 3. Create comment-weighted pairs, shuffle, and train
Every valid comment is retained before the trainer's subsampling. Blank authors are excluded and counted in the run metadata. When reading the exported CSV, unmatched articles cause an error. With raw shards, comments outside the selected article IDs are excluded. Topic and author tokens are integer codes, so spaces or separators in labels cannot corrupt the trainer input. Raw user IDs are not written to the final embedding outputs.

**Input coverage:** The supplied `comments_top_1000.csv` has valid comments for only 945 selected topics. The default therefore streams all 120 raw comment shards and retains the selected article IDs. Set `USE_RAW_COMMENTS = False` only after regenerating a complete export; missing-topic coverage will raise an error.


In [ ]:
def comment_chunks():
    if USE_RAW_COMMENTS:
        paths = sorted(RAW_DATA_DIR.glob("comments_file_*_[0-9][0-9].tsv"))
        if not paths:
            raise FileNotFoundError("No raw comment shards found")
    else:
        paths = [DATA_DIR / "comments_top_1000.csv"]
    for path in paths:
        with pd.read_csv(path, sep="\t" if USE_RAW_COMMENTS else ",", dtype="string",
                         keep_default_na=False, usecols=["Article_Id", "Author_Id"],
                         chunksize=CHUNK_SIZE) as reader:
            yield from reader


def train_embeddings():
    rng = np.random.default_rng(SHUFFLE_SEED)
    user_codes = {}
    user_counts = Counter()
    topic_counts = np.zeros(len(topics_df), dtype=np.int64)
    total_pairs = blank_authors = 0
    with TemporaryDirectory(prefix="sgns-", dir=OUTPUT_DIR) as scratch:
        scratch = Path(scratch)
        binary_pairs = scratch / "pairs.bin"
        with binary_pairs.open("wb") as output:
            for chunk in comment_chunks():
                topic_ids = chunk["Article_Id"].map(article_to_topic)
                if not USE_RAW_COMMENTS and topic_ids.isna().any():
                    raise ValueError("Exported comments have unmatched Article_Id values")
                # Raw shards also contain articles outside the selected topics.
                selected = topic_ids.notna()
                chunk = chunk.loc[selected]
                topic_ids = topic_ids.loc[selected]
                valid = chunk["Author_Id"].str.strip().ne("")
                blank_authors += int((~valid).sum())
                authors = chunk.loc[valid, "Author_Id"]
                for author in authors.unique():
                    if author not in user_codes:
                        user_codes[author] = len(user_codes)
                users = authors.map(user_codes).to_numpy(dtype=np.int32)
                topic_ids = topic_ids.loc[valid].to_numpy(dtype=np.int32)
                np.column_stack((topic_ids, users)).astype(np.int32).tofile(output)
                topic_counts += np.bincount(topic_ids, minlength=len(topics_df))
                ids, counts = np.unique(users, return_counts=True)
                user_counts.update(dict(zip(ids.tolist(), counts.tolist())))
                total_pairs += len(users)
        if (topic_counts == 0).any() or len(user_codes) < 2:
            raise ValueError("Every topic needs training comments and at least two authors overall")
        n_users = len(user_codes)
        del user_codes
        pairs = np.memmap(binary_pairs, dtype=np.int32, mode="r+", shape=(total_pairs, 2))
        # Shuffle packed 8-byte records through NumPy's fast 1D path.
        # Each topic/author pair stays intact; the backing storage remains disk-mapped.
        rng.shuffle(np.asarray(pairs).view(np.uint64).reshape(-1))
        pairs.flush()
        with (scratch / "pairs.txt").open("w") as output:
            # The upstream trainer skips the initial line while aligning thread offsets.
            output.write("\n")
            for start in range(0, total_pairs, CHUNK_SIZE):
                output.writelines(f"t{topic} u{user}\n" for topic, user in np.asarray(pairs[start:start + CHUNK_SIZE]))
        del pairs
        with (scratch / "topics.vocab").open("w") as output:
            output.writelines(f"t{i} {count}\n" for i, count in enumerate(topic_counts))
        with (scratch / "users.vocab").open("w") as output:
            output.writelines(f"u{i} {user_counts[i]}\n" for i in range(n_users))
        command = [str(trainer), "-train", "pairs.txt", "-wvocab", "topics.vocab",
                   "-cvocab", "users.vocab", "-output", str((OUTPUT_DIR / "topic_vectors.txt").resolve()),
                   "-size", str(DIMENSIONS), "-negative", str(NEGATIVE), "-alpha", str(ALPHA),
                   "-sample", str(SAMPLE), "-iters", str(EPOCHS), "-threads", str(THREADS),
                   "-binary", "0", "-debug", "0"]
        with (OUTPUT_DIR / "training.log").open("w") as log:
            subprocess.run(command, cwd=scratch, stdout=log, stderr=subprocess.STDOUT, check=True)
    return dict(training_pairs=total_pairs, users=n_users, excluded_blank_authors=blank_authors,
                topic_comment_counts=topic_counts.tolist())

training_stats = train_embeddings()

## 4. Load and save one vector per topic
`topic_embeddings_df` contains the labels and 150 **raw input/word-vector** coordinates, as used for communities in the paper. A separate L2-normalized array is provided for cosine comparisons. Rows in both arrays follow `topic_metadata.csv` and its `Topic_id` column. These embeddings alone do not define political or other social axes.

In [ ]:
vectors_by_token = {}
with (OUTPUT_DIR / "topic_vectors.txt").open() as source:
    rows, dimensions = map(int, source.readline().split())
    if (rows, dimensions) != (len(topics_df), DIMENSIONS):
        raise ValueError("Unexpected embedding dimensions")
    for line in source:
        token, *values = line.split()
        if token in vectors_by_token or len(values) != DIMENSIONS:
            raise ValueError("Malformed or duplicate embedding row")
        vectors_by_token[token] = np.asarray(values, dtype=np.float32)
if set(vectors_by_token) != {f"t{i}" for i in topics_df["Topic_id"]}:
    raise ValueError("Missing or unexpected topic vectors")
topic_embeddings = np.stack([vectors_by_token[f"t{i}"] for i in topics_df["Topic_id"]])
norms = np.linalg.norm(topic_embeddings, axis=1, keepdims=True)
if not np.isfinite(topic_embeddings).all() or (norms == 0).any():
    raise ValueError("Nonfinite or zero topic vectors")
normalized_topic_embeddings = topic_embeddings / norms
topic_embeddings_df = pd.concat([
    topics_df[["Topic_id", *TOPIC_COLUMNS]],
    pd.DataFrame(topic_embeddings, columns=[f"embedding_{i:03d}" for i in range(DIMENSIONS)])
], axis=1)
topics_df.to_csv(OUTPUT_DIR / "topic_metadata.csv", index=False)
topic_embeddings_df.to_csv(OUTPUT_DIR / "topic_embeddings.csv", index=False)
np.save(OUTPUT_DIR / "topic_embeddings.npy", topic_embeddings)
np.save(OUTPUT_DIR / "topic_embeddings_normalized.npy", normalized_topic_embeddings)
run_metadata = dict(paper="https://doi.org/10.1038/s41586-021-04167-x",
                    implementation="https://github.com/BIU-NLP/word2vecf", commit=actual_commit,
                    dimensions=DIMENSIONS, negative=NEGATIVE, alpha=ALPHA, sample=SAMPLE,
                    epochs=EPOCHS, threads=THREADS, shuffle_seed=SHUFFLE_SEED,
                    selection="top 1000 leaf topics by unique users",
                    comment_source="raw TSV shards" if USE_RAW_COMMENTS else "selected CSV", **training_stats)
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")